In [ ]:
%%capture
from pathlib import Path

from dj_notebook import activate

plus = activate(dotenv_file="/Users/erikvw/source/edc_source/meta-edc/.env")
report_folder = Path("/Users/erikvw/Documents/ucl/protocols/meta3/reports/")
export_folder = Path("/Users/erikvw/Documents/ucl/protocols/meta3/export/")

In [ ]:
import pandas as pd
from django_pandas.io import read_frame
from django.conf import settings
from edc_lab_results_import.models import Result
from edc_lab_results_import.result_importer import ResultImporter
from edc_identifier.utils import is_valid_subject_identifier
from edc_lab_panel.panels import wbc_differential
from meta_labs.dataframe import get_df_bloodresults
from edc_model_to_dataframe.constants import SYSTEM_COLUMNS



In [ ]:
df_bloodresults = get_df_bloodresults()

In [ ]:
# 126.10
df_bloodresults["result_value"] = df_bloodresults["result_value"].astype("float")
# df_bloodresults
df_bloodresults.source.value_counts()

In [ ]:
df = ResultImporter.model_to_dataframe()

In [ ]:
open_datetime = settings.EDC_PROTOCOL_STUDY_OPEN_DATETIME
df[(df["result_datetime"]>=open_datetime) & (df["subject_identifier"].isna()) & (df["screening_identifier"].isna())][["name_id", "result_datetime"]].groupby("name_id").size()

In [ ]:
# df[df["name_id"]=="AMN/105-30-0016-9"]
df[df["sample_no"]=="0123558710/0"][["name_id", 'result_datetime', "report_type", "source_utestid", "result_value"]]


In [ ]:
df.query("~subject_identifier.isna() and visit_code==''")

In [ ]:
path = "/Users/erikvw/upload/gmail/results_importer_202607240115.parquet"
df = pd.read_parquet(path)

In [ ]:
df

In [ ]:
importer = ResultImporter(
    "MNH",
    Path("~/upload/testpdf").expanduser(),
    is_valid_identifier_func=is_valid_subject_identifier,
    extra_panels=[wbc_differential],
)

In [ ]:
importer.run(to_model=True, df_to_path=Path("/tmp/testpdf"))

In [ ]:
importer.model_to_dataframe()



In [ ]:
from parse_trial_labs.parsers.parse_mnh import parser as p
import inspect
print(p.__file__)
print(inspect.getsource(p.parse_name_id))
print(importer.is_valid_identifier_func is is_valid_subject_identifier)
print(importer.is_valid_identifier_func)

In [ ]:
importer.run(to_model=False, df_to_path=Path("/tmp"))
# importer.df.to_parquet("/tmp/tmp.parquet", index=False)

In [ ]:
"""
Files with duplicate results collapsed:
  AAM AMANA.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  ABM_1.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC ACID
  AGC - HINDU.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  AGC HIND1.pdf: ALBUMIN, AMYLASE, CREATININE, UREA NITROGEN, URIC ACID
  AIA HINDU.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  ASK AMANA.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  BMM_10.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC ACID
  DALSHAC-MVUY CHEM.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC AC$
  EEN105-10-0095-6     chem.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  FHK.pdf: URIC ACID
  MMA_5.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC ACID
  PJM TEMEKE.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN
  PJM_7.pdf: URIC ACID
  RWB_3.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC ACID
  SBG.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CREATININE, GAMMA GT, UREA NITROGEN, URIC ACID
  VAH1.pdf: ALBUMIN, ALKALINE PHOSPHATASE, ALT(SGPT), AMYLASE, AST(SGOT), CHOLESTEROL, CREATININE, GAMMA GT, HDL CHOLESTEROL, LDL CHOL (CALC), TRIGLYCERIDES, UREA NITROGEN, URIC ACID
"""

In [ ]:
fname = "/tmp/results_importer_202607151936.parquet"
importer.df = pd.read_parquet(fname, engine="pyarrow")
len(importer.df)


In [ ]:
importer.resolve()
len(importer.df)


In [ ]:
len(importer.df)

In [ ]:
importer.dataframe_to_model()


In [ ]:
importer.df.site.value_counts()

In [ ]:
importer.df.to_parquet("/tmp/tmp.parquet", index=False)
df_orig = importer.df.copy()
# df_orig.to_csv("/tmp/tmpraw.csv", index=False)

In [ ]:
# importer.df = pd.read_csv("/tmp/tmpraw.csv")

In [ ]:
importer.dataframe_to_model(importer.df, batch_size=1)
len(importer.df)


In [ ]:
df_bloodresults = get_df_bloodresults()
len(df_bloodresults)

In [ ]:
importer.df.site.value_counts()

In [ ]:
df_bloodresults.duplicated(subset=["requisition", "utestid"]).any()

In [ ]:
cols1 = [
    "order_no",
    "result_no",
    "sample_no",
    "result_status",
    "source_utestid",
    "utestid",
    "result_datetime",
    "name_id",
]
cols2 = ["result_status", "requisition", "utestid"]
importer.df.duplicated(subset=cols1).any()


In [ ]:
fields = ",".join(cols1)
f"select {fields}, count(*) from edc_lab_results_import_result group by {fields} having count(*)>1;"

In [ ]:
col1 = [
    "subject_identifier",
    "screening_identifier",
    "source_utestid",
    "utestid",
    "source_units",
    "units",
    "report_type",
    "result_status",
    "order_no",
    "sample_no",
    "result_no",
    "name_id",
]
col2 = ["order_no","sample_no","result_no","report_type","subject_visit", "requisition", "result_status", "utestid", "subject_identifier"]

importer.df.drop_duplicates(subset=col2, keep=False)

In [ ]:
from decimal import Decimal
x = 1.223492e+09
Decimal(str(x))

In [ ]:
cols = ['result',
 'source_file',
 'source_utestid',
 'visit_datetime',
 'specimen_received_by',
 'sample_type',
 'sample_condition',
 'name_id',
 'subject_identifier',
 'flag',
 'result_no',
 'schedule_name',
 'order_no',
 'subject_visit',
 'specimen_collected_datetime',
 'priority',
 'report_type',
 'drawn_datetime',
 'source_units',
 'reported_by',
 'specimen_collected_by',
 'age',
 'requisition',
 'units',
 'subject_visit_right',
 'order_datetime',
 'requisition_identifier',
 'ordered_by',
 'sample_no',
 'visit_code_sequence',
 'clinic_ward',
 'reference_range_lower',
 'requisition_datetime',
 'verified_by',
 'specimen_received_datetime',
 'panel_name',
 'visit_code',
 'site',
 'result_status',
 'verified_datetime',
 'screening_identifier',
 'reference_range_upper',
 'utestid',
 'sex',
 'reported_datetime',
 'result_datetime']

In [ ]:
importer.df.duplicated(subset=cols, keep=False).any()

In [ ]:
importer.df.duplicated(subset=["requisition", "utestid"], keep=False).any()

In [ ]:
len(importer.df)

In [ ]:
df_results = read_frame(Result.objects.all(), verbose=False)

In [ ]:
len(df_results)

In [ ]:
importer = ResultImporter(
    "MNH",
    Path("~/upload/gmail").expanduser(),
    is_valid_identifier_func=is_valid_subject_identifier,
    extra_panels=[wbc_differential],
)
fname = "/tmp/results_importer_202607151936.parquet"
importer.df = pd.read_parquet(fname, engine="pyarrow")
df_results = importer.model_to_dataframe()
df_bloodresults = get_df_bloodresults()
df_requisitions = importer.df_requisitions.copy()


In [ ]:
len(df_requisitions)

In [ ]:
cols = ["subject_identifier", "visit_datetime", "visit_code", "visit_code_sequence", "requisition", "subject_visit", "requisition_identifier", "panel_name", "drawn_datetime"]

df_missing = df_requisitions[cols].merge(df_bloodresults[[c for c in df_bloodresults if c not in ["panel_name"]]], on=["requisition", "subject_visit"], how="left", suffixes=["", "_y"]).query("source.isna()").copy().reset_index(drop=True)
len(df_missing)

In [ ]:
df = (
    df_missing[cols]
    .merge(df_results[[c for c in df_results.columns if c not in SYSTEM_COLUMNS]], on=["requisition", "subject_visit"], how="left", suffixes=["", "_y"])
    .query("id.isna() and panel_name != 'blood_glucose'").drop_duplicates(subset=cols, keep="first")
    .sort_values(["subject_identifier", "visit_datetime", "visit_code", "visit_code_sequence","panel_name"])
)[[c for c in cols if c not in ["requisition", "subject_visit"]]].copy().reset_index(drop=True)

df.to_csv(report_folder / "requisition_without_results_20260719.csv", index=False, date_format="%Y-%m-%d")

In [ ]:
pivoted = df.pivot_table(
    index=[
        "subject_identifier",
        "visit_datetime",
        "visit_code",
        "visit_code_sequence",
    ],
    columns="panel_name",
    values="drawn_datetime",
    aggfunc="first",
).reset_index()
pivoted.to_csv(report_folder / "requisition_without_results_pivot_20260719.csv", index=False, date_format="%Y-%m-%d")


In [ ]:
pivoted

In [ ]:
df = df_bloodresults.merge(df_results[["requisition", "utestid", "result_value"]], on=["requisition", "utestid"], how="left", suffixes=["", "_y"])

In [ ]:

df_requisitions["visit_code"] = df_requisitions["visit_code"].astype("string").fillna(pd.NA)
df_requisitions["panel_name"] = df_requisitions["panel_name"].astype("string").fillna(pd.NA)


In [ ]:
df_match = df[df["result_value_y"].isna()].copy().reset_index(drop=True)

df_match.merge(df_requisitions, on=["requisition", "panel_name", "utestid"], how="left", suffixes=["", "_z"])

In [ ]:
df_requisitions.requisition.dtype

In [ ]:
df_results.dtypes


In [ ]:
# calculate which visits have filled reqisitions but unfilled bloodresults

